In [0]:
%sql
USE dbjahan.bronze_data;
CREATE OR REPLACE TABLE  dbjahan.bronze_data.elpris_raw (
  elomrade     STRING,
  tid_start    STRING,
  tid_slut     STRING,
  sek_per_kwh  DOUBLE,
  hamtad_tid   TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DESCRIBE TABLE EXTENDED dbjahan.bronze_data.bilar_trafik_raw


In [0]:
import requests
import time
from datetime import datetime, timedelta
from pyspark.sql import Row

def hamta(api_namn, url, params=None, timeout=90):
    senaste = None
    for forsok in range(1, 4):
        try:
            svar = requests.get(url, params=params, timeout=timeout)
            if svar.status_code == 200:
                return svar
            senaste = str(svar.status_code)
        except requests.RequestException as fel:
            senaste = str(fel)
        if forsok < 3:
            time.sleep(5)
    raise RuntimeError(
        f"{api_namn} svarade {senaste} efter 3 försök. URL: {url}"
    )

zoner = ["SE1", "SE2", "SE3", "SE4"]
start = datetime(2025, 1, 1)
slut = datetime(2025, 12, 31)

rader = []
dag = start
while dag <= slut:
    for zon in zoner:
        url = (
            f"https://www.elprisetjustnu.se/api/v1/prices/"
            f"{dag:%Y}/{dag:%m-%d}_{zon}.json"
        )
        r = hamta(
            f"Elpriset just nu för {zon} {dag:%Y-%m-%d}",
            url,
            timeout=30,
        )
        for rad in r.json():
            pris = rad.get("SEK_per_kWh")
            rader.append(Row(
                elomrade=zon,
                tid_start=rad.get("time_start"),
                tid_slut=rad.get("time_end"),
                sek_per_kwh=float(pris) if pris is not None else None,
                hamtad_tid=datetime.now(),
            ))
    dag += timedelta(days=1)

df = spark.createDataFrame(rader)
df.createOrReplaceTempView("elpris_nya")

spark.sql("""
INSERT INTO dbjahan.bronze_data.elpris_raw
SELECT n.elomrade, n.tid_start, n.tid_slut, n.sek_per_kwh, n.hamtad_tid
FROM elpris_nya n
WHERE NOT EXISTS (
  SELECT 1
  FROM dbjahan.bronze_data.elpris_raw t
  WHERE t.elomrade = n.elomrade
    AND t.tid_start = n.tid_start
)
""")
print("Klart elpris", df.count())

Klart elpris 61548


In [0]:
%sql
SELECT * FROM dbjahan.bronze_data.elpris_raw

elomrade,tid_start,tid_slut,sek_per_kwh,hamtad_tid
SE4,2025-12-11T20:30:00+01:00,2025-12-11T20:45:00+01:00,0.64541,2026-09-28T12:39:39.421Z
SE4,2025-12-11T20:45:00+01:00,2025-12-11T21:00:00+01:00,0.62565,2026-09-28T12:39:39.421Z
SE4,2025-12-11T21:00:00+01:00,2025-12-11T21:15:00+01:00,0.62826,2026-09-28T12:39:39.421Z
SE4,2025-12-11T21:15:00+01:00,2025-12-11T21:30:00+01:00,0.60535,2026-09-28T12:39:39.421Z
SE4,2025-12-11T21:30:00+01:00,2025-12-11T21:45:00+01:00,0.57159,2026-09-28T12:39:39.421Z
SE4,2025-12-11T21:45:00+01:00,2025-12-11T22:00:00+01:00,0.5336,2026-09-28T12:39:39.421Z
SE4,2025-12-11T22:00:00+01:00,2025-12-11T22:15:00+01:00,0.56617,2026-09-28T12:39:39.421Z
SE4,2025-12-11T22:15:00+01:00,2025-12-11T22:30:00+01:00,0.52199,2026-09-28T12:39:39.421Z
SE4,2025-12-11T22:30:00+01:00,2025-12-11T22:45:00+01:00,0.48932,2026-09-28T12:39:39.421Z
SE4,2025-12-11T22:45:00+01:00,2025-12-11T23:00:00+01:00,0.47608,2026-09-28T12:39:39.421Z
